# 🔥 Tabular Data Classification with PyTorch — Rice Type Classification (Corrected & Annotated)

This is a cleaned-up, bug-fixed, and heavily annotated version of your `Untitled0.ipynb`. It builds a **binary classification** neural network on the [Rice Type Classification dataset](https://www.kaggle.com/datasets/mssmartypants/rice-type-classification) (Kaggle) — a step up in complexity from the previous notebooks in this series, since it introduces a real `Dataset`/`DataLoader` pipeline instead of feeding whole tensors straight into the model.

### What you'll find here
- 📝 A markdown explanation **before every code cell**, with reference tables for every built-in function used
- 🎁 The model is also shown built the easy way, with **`nn.Sequential`**, right alongside the original hand-written `nn.Module` subclass
- 📋 A full **cheat-sheet** at the very end for quick revision

### The workflow this notebook follows
```
1. Download the dataset from Kaggle
2. Clean and normalize it with pandas
3. Split it into train / validation / test sets
4. Wrap the arrays in a PyTorch Dataset + DataLoader
5. Build a small feed-forward neural network
6. Pick a loss function + optimizer
7. Train, validating after every epoch
8. Evaluate once on the untouched test set
9. Plot loss/accuracy curves
10. Run a live, manual prediction
```

> 💡 **Tip:** Run cells top to bottom. Cell 0 will interactively prompt you for your Kaggle username and API key the first time it runs (see the setup section below) — that's expected.


---
## 1. Download the dataset

| Function | What it does |
|---|---|
| `opendatasets.download(url)` | downloads a Kaggle (or Google Drive) dataset directly into your notebook's working directory. The **first time** you run it, it interactively prompts for your Kaggle username and API key (from `kaggle.json` — generate one at kaggle.com → Account → "Create New API Token") |

> **🔧 Security note:** the original notebook had a follow-up cell containing a string that looked like a pasted API credential. That's been removed here. Never leave real credentials in a notebook cell (even "commented out") — if you share this notebook or push it to GitHub, treat any key that was ever pasted into it as compromised and regenerate it from your Kaggle account settings.


In [ ]:
#Tabular data classification

!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

## 2. Imports

| Import | Purpose |
|---|---|
| `torch`, `torch.nn` | tensors, autograd, model building blocks |
| `torch.optim.Adam` | the optimizer we'll train with |
| `torch.utils.data.Dataset, DataLoader` | data pipeline base classes (we end up using the ready-made `TensorDataset` instead of subclassing `Dataset` directly — see §9) |
| `torchsummary.summary` | prints a Keras-style layer-by-layer model summary |
| `sklearn.model_selection.train_test_split` | splits arrays into train/test subsets |
| `sklearn.metrics.accuracy_score` | computes accuracy from true vs. predicted labels (used in §16 as a cross-check) |
| `matplotlib.pyplot`, `pandas`, `numpy` | plotting, tabular data handling, array math |


In [ ]:
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## 3. Device-agnostic code

| Function | What it does |
|---|---|
| `torch.cuda.is_available()` | `True` if a CUDA GPU is visible to PyTorch |


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

---
## 4. Load and clean the data

| Function | What it does |
|---|---|
| `pd.read_csv(path)` | reads a CSV file into a `DataFrame` |
| `DataFrame.head(n=5)` | previews the first `n` rows |
| `DataFrame.dropna(inplace=True)` | removes rows containing missing values, modifying the DataFrame in place |
| `DataFrame.drop(columns, axis=1, inplace=True)` | removes column(s) — `axis=1` means "columns", not "rows" |


In [ ]:
df = pd.read_csv("/content/rice-type-classification/riceClassification.csv")
df.head()

In [ ]:
df.dropna(inplace=True)
df.drop(['id'], axis=1, inplace=True)  # the id column carries no predictive information
print(df.shape)

In [ ]:
df

## 5. Normalize the features

Each feature column is divided by its own maximum absolute value, scaling every column into roughly `[0, 1]` (all of these physical measurements — area, perimeter, etc. — are positive). Neural networks train much better when input features are on a similar scale.



In [ ]:
original_df = df.copy()  # keep the un-normalized values around for later (used in the live prediction demo)

feature_columns = df.columns[:-1]  # everything except the last column ("Class", the label)
for column in feature_columns:
    df[column] = df[column] / df[column].abs().max()

df.head()

## 6. Separate features (X) from the label (Y)

| Function | What it does |
|---|---|
| `np.array(df.iloc[:, :-1])` | converts every column **except the last** into a NumPy array — our features |
| `np.array(df.iloc[:, -1])` | converts just the **last** column into a NumPy array — our label |


In [ ]:
X = np.array(df.iloc[:, :-1])  # all columns except the last (the label)
Y = np.array(df.iloc[:, -1])   # just the last column (the label)

---
## 7. First split: train vs. everything else

| Function | Signature | What it does |
|---|---|---|
| `train_test_split(X, Y, test_size, random_state)` | from `sklearn.model_selection` | randomly splits arrays; `test_size=0.3` reserves 30% for `X_test`/`y_test`, `random_state` makes it reproducible |


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## 8. Second split: carve the held-out 30% into validation + test


In [ ]:
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
X_val.shape, X_test.shape, y_val.shape, y_test.shape

---
## 9. Wrap the NumPy arrays in a PyTorch `Dataset`

`DataLoader` (used a few cells down) needs a `Dataset` object, not raw NumPy arrays — it's what lets PyTorch batch, shuffle, and iterate over your data efficiently. Rather than subclassing `Dataset` by hand, we use PyTorch's ready-made **`TensorDataset`**, which wraps tensors directly.

| Class / function | What it does |
|---|---|
| `torch.tensor(data, dtype=...)` | creates a tensor from a NumPy array/list with an explicit dtype (see the `torch.tensor` vs `torch.Tensor` note in §17 for why the explicit `dtype` matters) |
| `Tensor.view(-1, 1)` | reshapes a 1-D tensor of shape `[N]` into a column vector of shape `[N, 1]`, without copying data |
| `TensorDataset(*tensors)` | a built-in `Dataset` that simply indexes into the tensors you pass it — `dataset[i]` returns `(X[i], y[i])` |




In [ ]:
def dataset(X_data, y_data):
    X_tensor = torch.tensor(X_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data, dtype=torch.float32).view(-1, 1)  # shape [N] -> [N, 1]
    return torch.utils.data.TensorDataset(X_tensor, y_tensor)

In [ ]:
training_data = dataset(X_train, y_train)
testing_data = dataset(X_test, y_test)
validation_data = dataset(X_val, y_val)

## 10. Hyperparameters

Collecting these as named constants (rather than scattering literal numbers through the code) makes them easy to find and experiment with later.


In [ ]:
BATCH_SIZE = 32
EPOCHS = 10
HIDDEN_NEURONS = 10
LR = 1e-3

## 11. Create `DataLoader`s

| Function | Signature | What it does |
|---|---|---|
| `DataLoader(dataset, batch_size, shuffle)` | from `torch.utils.data` | serves up a `Dataset` in mini-batches of `batch_size`; `shuffle=True` reshuffles the order every epoch |




In [ ]:
train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)
testing_dataloader = DataLoader(testing_data, batch_size=BATCH_SIZE, shuffle=False)

---
## 12. Build the model

Architecture: `10 features → 10 hidden units → 1 output`, ending in a `Sigmoid` so the model outputs a probability directly (matching `nn.BCELoss` in §13, which expects probabilities — not logits, unlike the `BCEWithLogitsLoss` used in the classification notebook earlier in this series).


| Class | What it does |
|---|---|
| `nn.Linear(in, out)` | fully-connected layer, `y = xW^T + b` |
| `nn.ReLU()` | `max(0, x)` — the non-linearity that lets stacked layers learn more than a plain linear model |
| `nn.Sigmoid()` | squashes a value into `[0, 1]`, read as "probability of class 1" |


In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        self.input_layer = nn.Linear(X.shape[1], HIDDEN_NEURONS)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(HIDDEN_NEURONS, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.input_layer(x)
        x = self.relu(x)
        x = self.linear(x)
        x = self.sigmoid(x)
        return x

model = MyModel().to(device)
model

## 12.1 🎁 The same model, the easy way — `nn.Sequential`

You asked for this as an addition — it wasn't in your original notebook. Writing a custom `__init__`/`forward()` (above) is the flexible, general-purpose way to build a model, but for a model that's just "one layer after another with nothing branching or looping", `nn.Sequential` is the easiest way to write the exact same thing: pass it a list of layers in order, and it chains them for you — no `forward()` method needed at all.

```python
nn.Sequential(layer1, layer2, layer3, ...)
```
calling `model(x)` simply runs `x` through each layer in the order given, feeding each layer's output into the next.

> This cell **replaces** `model` — from here on (summary, training, live prediction), `model` refers to this `nn.Sequential` version. It's architecturally identical to `MyModel` above (same layers, same order), just less code to write and read.


In [ ]:
model = nn.Sequential(
    nn.Linear(X.shape[1], HIDDEN_NEURONS),
    nn.ReLU(),
    nn.Linear(HIDDEN_NEURONS, 1),
    nn.Sigmoid()
).to(device)

model

## 12.2 View the model summary

| Function | Signature | What it does |
|---|---|---|
| `summary(model, input_size)` | from `torchsummary` | prints a Keras-style table of every layer, its output shape, and parameter count. `input_size` excludes the batch dimension |


In [ ]:
summary(model, (X.shape[1],))

---
## 13. Loss function & optimizer

| Class | What it does |
|---|---|
| `nn.BCELoss()` | Binary Cross-Entropy loss — expects the model's output to already be a probability in `[0, 1]` (i.e. the model must end in `Sigmoid`, which ours does) |
| `torch.optim.Adam(params, lr)` | Adam optimizer — adapts the learning rate per-parameter; a strong general-purpose default |

> Since this model already ends in `Sigmoid`, `nn.BCELoss` is the *correct* pairing (unlike the classification notebook earlier in this series, where the fix was to remove `Sigmoid` and use `BCEWithLogitsLoss` instead). Both patterns are valid — just don't mix them (sigmoid + `BCEWithLogitsLoss`, or raw logits + `BCELoss`, would both be wrong).


In [ ]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(lr=LR, params=model.parameters())

---
## 14. Training and validation loop

The standard 5-step training loop, run once per batch, plus a no-grad validation pass after each epoch.


| Function | What it does |
|---|---|
| `Tensor.squeeze(1)` | drops dimension 1 if it has size 1 — turns the model's `[batch, 1]` output (and the `[batch, 1]` labels) into `[batch]`, so they match for the loss/accuracy calculations |
| `Tensor.round()` | rounds each probability to `0.` or `1.` |
| `torch.eq(a, b)` | element-wise equality — combined with `.sum().item()`, counts how many predictions matched the true labels in a batch |
| `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` | the standard gradient reset → backprop → parameter-update sequence |
| `model.train()` / `model.eval()` | switch layers like Dropout/BatchNorm between training and inference behavior (no such layers here, but always good practice) |
| `torch.no_grad()` | context manager that disables gradient tracking, used during evaluation |


In [ ]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_acc_val = 0
    total_loss_val = 0

    ## Training
    model.train()
    for data in train_dataloader:
        # Move data to the same device as the model
        inputs, labels = data[0].to(device), data[1].to(device)

        prediction = model(inputs).squeeze(1)
        labels = labels.squeeze(1)

        batch_loss = loss_fn(prediction, labels)
        total_loss_train += batch_loss.item()

        acc = (prediction.round() == labels).sum().item()
        total_acc_train += acc

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

    ## Validation
    model.eval()
    with torch.no_grad():
        for data in validation_dataloader:
            inputs, labels = data[0].to(device), data[1].to(device)

            prediction = model(inputs).squeeze(1)
            labels = labels.squeeze(1)

            batch_loss = loss_fn(prediction, labels)
            total_loss_val += batch_loss.item()

            acc = (prediction.round() == labels).sum().item()
            total_acc_val += acc

    total_loss_train_plot.append(round(total_loss_train / len(train_dataloader), 4))
    total_loss_validation_plot.append(round(total_loss_val / len(validation_dataloader), 4))
    total_acc_train_plot.append(round(total_acc_train / len(training_data) * 100, 4))
    total_acc_validation_plot.append(round(total_acc_val / len(validation_data) * 100, 4))

    print(f'''Epoch no. {epoch + 1} Train Loss: {total_loss_train/len(train_dataloader):.4f} Train Accuracy: {(total_acc_train/len(training_data)*100):.4f} Validation Loss: {total_loss_val/len(validation_dataloader):.4f} Validation Accuracy: {(total_acc_val/len(validation_data)*100):.4f}''')
    print("=" * 50)

Now that the leakage bug is fixed, expect validation accuracy to look a little lower (and more realistic) than it did before — that's the fix working as intended, not a regression.


---
## 15. Final evaluation on the untouched test set



In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    total_loss_test = 0
    total_acc_test = 0
    for data in testing_dataloader:
        inputs, labels = data[0].to(device), data[1].to(device)

        prediction = model(inputs).squeeze(1)
        labels = labels.squeeze(1)

        batch_loss_test = loss_fn(prediction, labels)
        total_loss_test += batch_loss_test.item()

        acc = (prediction.round() == labels).sum().item()
        total_acc_test += acc

        all_preds.extend(prediction.round().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print(f"Test Loss: {total_loss_test/len(testing_dataloader):.4f}")
print(f"Accuracy Score is: {round((total_acc_test/len(testing_data))*100, 2)}%")
print(f"sklearn cross-check: {round(accuracy_score(all_labels, all_preds)*100, 2)}%")

---
## 16. Plot the training curves

| Function | What it does |
|---|---|
| `plt.subplots(nrows, ncols, figsize)` | creates a figure with a grid of subplots — here, 1 row × 2 columns, side by side |
| `ax.plot(values, label)` | plots a line, `label` feeds the legend |
| `ax.set_title` / `set_xlabel` / `set_ylabel` / `set_ylim` | axis labeling and scaling |
| `plt.tight_layout()` | auto-adjusts spacing so labels/titles don't overlap |


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

axs[0].plot(total_loss_train_plot, label='Training Loss')
axs[0].plot(total_loss_validation_plot, label='Validation Loss')
axs[0].set_title('Training and Validation Loss over Epochs')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].set_ylim([0, 2])
axs[0].legend()

axs[1].plot(total_acc_train_plot, label='Training Accuracy')
axs[1].plot(total_acc_validation_plot, label='Validation Accuracy')
axs[1].set_title('Training and Validation Accuracy over Epochs')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy')
axs[1].set_ylim([0, 100])
axs[1].legend()

plt.tight_layout()

---
## 17. Try a live prediction

Collects one raw sample from manual input, applies the **same** normalization used during training (dividing by each column's max from `original_df`), and runs it through the trained model.


| Function | What it does |
|---|---|
| `input(prompt)` | built-in Python function — pauses and waits for the user to type a value |
| `Tensor.unsqueeze(0)` | inserts a new size-1 dimension at position 0 — turns a single sample `[10]` into a batch of one, `[1, 10]` |
| `Tensor.item()` | extracts a plain Python number from a tensor containing exactly one element |


In [ ]:
area = float(input("Area: ")) / original_df['Area'].abs().max()
MajorAxisLength = float(input("Major Axis Length: ")) / original_df['MajorAxisLength'].abs().max()
MinorAxisLength = float(input("Minor Axis Length: ")) / original_df['MinorAxisLength'].abs().max()
Eccentricity = float(input("Eccentricity: ")) / original_df['Eccentricity'].abs().max()
ConvexArea = float(input("Convex Area: ")) / original_df['ConvexArea'].abs().max()
EquivDiameter = float(input("EquivDiameter: ")) / original_df['EquivDiameter'].abs().max()
Extent = float(input("Extent: ")) / original_df['Extent'].abs().max()
Perimeter = float(input("Perimeter: ")) / original_df['Perimeter'].abs().max()
Roundness = float(input("Roundness: ")) / original_df['Roundness'].abs().max()
AspectRation = float(input("AspectRation: ")) / original_df['AspectRation'].abs().max()

my_inputs = [area, MajorAxisLength, MinorAxisLength, Eccentricity, ConvexArea,
             EquivDiameter, Extent, Perimeter, Roundness, AspectRation]

print("=" * 20)
model_inputs = torch.tensor(my_inputs, dtype=torch.float32).unsqueeze(0).to(device)  # shape [1, 10]

model.eval()
with torch.inference_mode():
    prediction = model(model_inputs)

print(prediction)
print("Class is: ", round(prediction.item()))

---
# 📋 Cheat Sheet — every function/class used in this notebook

### Data acquisition & cleaning
| Function | Signature | What it does |
|---|---|---|
| `opendatasets.download` | `od.download(url)` | downloads a Kaggle/Google Drive dataset; prompts for Kaggle credentials on first use |
| `pd.read_csv` | `pd.read_csv(path)` | loads a CSV into a DataFrame |
| `DataFrame.dropna` | `df.dropna(inplace=True)` | drops rows with missing values |
| `DataFrame.drop` | `df.drop(columns, axis=1, inplace=True)` | drops column(s) |
| `DataFrame.copy` | `df.copy()` | makes an independent copy (so later edits don't affect the original) |
| `DataFrame.iloc` | `df.iloc[:, :-1]` | position-based row/column selection |

### Tensors & data pipeline
| Function | Signature | What it does |
|---|---|---|
| `np.array` | `np.array(x)` | converts to a NumPy array |
| `train_test_split` | `train_test_split(X, y, test_size, random_state)` | splits arrays into subsets — **remember to split off of your held-out set, not the original data, when creating a 3-way split** |
| `torch.tensor` | `torch.tensor(data, dtype=...)` | preferred way to create a tensor from data, with explicit dtype |
| `torch.Tensor` | `torch.Tensor(data)` | legacy constructor — avoid; `torch.Tensor(int)` creates uninitialized memory, a classic footgun |
| `Tensor.view` | `tensor.view(-1, 1)` | reshapes without copying data |
| `Tensor.unsqueeze` | `tensor.unsqueeze(dim)` | inserts a size-1 dimension |
| `Tensor.squeeze` | `tensor.squeeze(dim)` | removes a size-1 dimension |
| `TensorDataset` | `TensorDataset(X, y)` | wraps tensors as an indexable `Dataset` |
| `DataLoader` | `DataLoader(dataset, batch_size, shuffle)` | batches and (optionally) shuffles a `Dataset` |

### Building & training a model
| Function | Signature | What it does |
|---|---|---|
| `nn.Module` | base class | parent class for all models/layers |
| `nn.Linear` | `nn.Linear(in, out)` | fully-connected layer |
| `nn.ReLU` | `nn.ReLU()` | non-linear activation, `max(0, x)` — needed between stacked linear layers |
| `nn.Sigmoid` | `nn.Sigmoid()` | squashes to `[0, 1]`, a probability |
| `nn.Sequential` | `nn.Sequential(layer1, layer2, ...)` | chains layers in order, no custom `forward()` needed — the easiest way to build a simple feed-forward model |
| `torchsummary.summary` | `summary(model, input_size)` | prints a per-layer parameter/shape summary |
| `nn.BCELoss` | `nn.BCELoss()` | binary cross-entropy loss; expects probabilities (model must end in `Sigmoid`) |
| `torch.optim.Adam` | `Adam(params, lr)` | adaptive per-parameter learning-rate optimizer |
| `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` | — | the standard gradient update sequence |
| `model.train()` / `model.eval()` | — | switch training/inference mode |
| `torch.no_grad()` / `torch.inference_mode()` | context managers | disable gradient tracking during evaluation |
| `torch.eq(a, b)` | — | element-wise equality check |
| `Tensor.round()` | — | rounds each element to the nearest integer |
| `Tensor.item()` | — | extracts a single Python number from a 1-element tensor |
| `accuracy_score` | `accuracy_score(y_true, y_pred)` | sklearn's accuracy metric — handy as an independent cross-check |

### Visualization
| Function | What it does |
|---|---|
| `plt.subplots(nrows, ncols, figsize)` | creates a grid of subplots |
| `ax.plot`, `.set_title`, `.set_xlabel`, `.set_ylabel`, `.set_ylim`, `.legend()` | standard per-axis plot styling |
| `plt.tight_layout()` | fixes overlapping labels/titles |

### Misc Python
| Function | What it does |
|---|---|
| `input(prompt)` | reads a line of user input as a string |

---
### 📚 References
- [Rice Type Classification dataset — Kaggle](https://www.kaggle.com/datasets/mssmartypants/rice-type-classification)
- [opendatasets — GitHub](https://github.com/JovianHQ/opendatasets)
- [01. PyTorch Workflow Fundamentals — learnpytorch.io](https://www.learnpytorch.io/01_pytorch_workflow/)
- [02. PyTorch Neural Network Classification — learnpytorch.io](https://www.learnpytorch.io/02_pytorch_classification/)
- [PyTorch documentation (v2.13)](https://docs.pytorch.org/docs/2.13/index.html)
- [scikit-learn: train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
